# NB13 — EBM Hyperparameter Robustness Check

**Purpose:** Address the critique that EBM's fixed hyperparameters (max_rounds=5000,
interactions=5, learning_rate=0.01, min_samples_leaf=2 — see NB05B) were never subjected
to a sensitivity check, unlike M3's isotonic calibration tuning. This notebook re-runs the
identical Stage 2 expanding-window CV (M2.5-matched, 41 macro features) under 5 alternative
hyperparameter configurations, varying interaction depth and learning rate — the two
parameters the critique named specifically.

**This notebook is a standalone extension.** It does not modify or retrain M2, M2.5-matched,
or M3. It reuses NB05B's exact CV function, bootstrap helper, and feature set for direct
comparability. Only the EBM hyperparameters change across runs.

**Inputs:** `df_sent_augmented.csv`, `M2_features.csv` (from NB05A) — read-only.


## Cell 1 — Imports and paths

In [ ]:
import os, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from math import isnan as import_isnan

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, average_precision_score as _aps
)
from sklearn.utils import resample

from interpret.glassbox import ExplainableBoostingClassifier

BASE    = Path(r'.')
OUT_DIR = BASE / 'data' / 'processed' / 'augmented_analysis'
FIG_DIR = BASE / 'figures' / 'augmented_analysis'

TARGET       = 'target_h2'
RANDOM_STATE = 42    # confirmed identical to NB05A/NB05B
N_BOOTSTRAP  = 1000  # confirmed identical to NB05B

print('Paths configured.')
print(f'OUT_DIR : {OUT_DIR}')
print(f'Exists  : {OUT_DIR.exists()}')


## Cell 2 — Load data and feature list (41-feature set, matching NB05B exactly)

In [ ]:
df_sent = pd.read_csv(OUT_DIR / 'df_sent_augmented.csv')
M2_FEATURES = pd.read_csv(OUT_DIR / 'M2_features.csv', header=None)[0].tolist()
EBM_FEATURES = M2_FEATURES  # identical alias to NB05B — 41 macro features only

n_crisis_sent  = int(df_sent[TARGET].sum())
base_rate_sent = df_sent[TARGET].mean()

print("=== DATASET (Stage 2, 41 macro features — matches NB05B's M2.5-matched exactly) ===")
print(f'Rows          : {len(df_sent)}')
print(f'Crisis events : {n_crisis_sent}  ({100*base_rate_sent:.2f}% base rate)')
print(f'Features      : {len(EBM_FEATURES)}')
assert len(EBM_FEATURES) == 41, f'Expected 41 features, found {len(EBM_FEATURES)}'
print('✅  Feature set verified: 41 macro features, identical to NB05B M2.5-matched.')


## Cell 3 — CV function and bootstrap helper (verbatim copy from NB05B)

In [ ]:
def expanding_window_cv(df, feature_cols, target_col, model_factory,
                        min_train_years=4, hold_out_years=(2019, 2020)):
    """Identical implementation to NB05B — ensures results are directly comparable."""
    cv_years   = sorted(y for y in df['year'].unique() if y not in hold_out_years)
    first_eval = cv_years[min_train_years]
    eval_years = [y for y in cv_years if y >= first_eval]

    oos_idx, oos_prob, fold_records = [], [], []
    fold_auprcs = []

    for yr in eval_years:
        tr = df['year'] < yr
        te = df['year'] == yr
        X_tr, y_tr = df.loc[tr, feature_cols], df.loc[tr, target_col]
        X_te, y_te = df.loc[te, feature_cols], df.loc[te, target_col]
        if y_tr.sum() == 0 or len(X_te) == 0:
            continue

        n_neg_fold = (y_tr == 0).sum()
        n_pos_fold = max((y_tr == 1).sum(), 1)
        fold_spw   = n_neg_fold / n_pos_fold

        mdl = model_factory(fold_spw)
        mdl.fit(X_tr.values, y_tr.values)

        prob = mdl.predict_proba(X_te.values)[:, 1]
        oos_idx.extend(df.index[te].tolist())
        oos_prob.extend(prob.tolist())

        fold_auprc = _aps(y_te, prob) if y_te.sum() > 0 else float('nan')
        if not import_isnan(fold_auprc):
            fold_auprcs.append(fold_auprc)

    y_oos = df.loc[oos_idx, target_col].values
    return y_oos, np.array(oos_prob), oos_idx, pd.DataFrame(fold_records)


def bootstrap_metrics(y_true, y_prob, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    """Bootstrap 95% CI for AUROC and AUPRC. Identical to NB05B."""
    rng = np.random.RandomState(seed)
    auroc_scores, auprc_scores = [], []
    for _ in range(n_boot):
        idx = resample(np.arange(len(y_true)), random_state=rng)
        if len(np.unique(y_true[idx])) < 2:
            continue
        auroc_scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
        auprc_scores.append(average_precision_score(y_true[idx], y_prob[idx]))
    return {
        'auroc_ci': (np.percentile(auroc_scores, 2.5), np.percentile(auroc_scores, 97.5)),
        'auprc_ci': (np.percentile(auprc_scores, 2.5), np.percentile(auprc_scores, 97.5)),
    }

print('CV and bootstrap functions ready (verbatim from NB05B).')


## Cell 4 — Define the 5 hyperparameter configurations

In [ ]:
class EBMWeighted:
    """Identical wrapper to NB05B — applies inverse-frequency sample weights at fit time."""
    def __init__(self, fold_spw, ebm_params):
        self.fold_spw   = fold_spw
        self.ebm_params = ebm_params
        self._model = ExplainableBoostingClassifier(**ebm_params)

    def fit(self, X, y):
        sample_weight = np.where(y == 1, self.fold_spw, 1.0)
        self._model.fit(X, y, sample_weight=sample_weight)
        return self

    def predict_proba(self, X):
        return self._model.predict_proba(X)


# ── 5 configurations: baseline (as reported in Ch4) + 4 alternatives ──────
# Varying exactly the two hyperparameters the critique named: interactions, learning_rate
CONFIGS = {
    'Baseline (Ch4 reported)': dict(max_rounds=5000, interactions=5, learning_rate=0.01,
                                     min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'Lower interactions (3)':  dict(max_rounds=5000, interactions=3, learning_rate=0.01,
                                     min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'Higher interactions (8)': dict(max_rounds=5000, interactions=8, learning_rate=0.01,
                                     min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'Lower learning rate (0.005)': dict(max_rounds=5000, interactions=5, learning_rate=0.005,
                                     min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'Higher learning rate (0.05)': dict(max_rounds=5000, interactions=5, learning_rate=0.05,
                                     min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
}

print(f'{len(CONFIGS)} configurations defined:')
for name, params in CONFIGS.items():
    print(f'  {name}: interactions={params["interactions"]}, learning_rate={params["learning_rate"]}')


## Cell 5 — Run CV under each configuration (SLOW — ~5x a single NB05B EBM fit)

In [ ]:
print('=== RUNNING 5 CONFIGURATIONS — this will take significantly longer than a single EBM fit ===')
print()

results = []
for name, params in CONFIGS.items():
    print(f'--- {name} ---')
    def factory(fold_spw, params=params):
        return EBMWeighted(fold_spw, params)

    y_oos, prob_oos, idx, folds = expanding_window_cv(df_sent, EBM_FEATURES, TARGET, factory)

    auroc = roc_auc_score(y_oos, prob_oos)
    auprc = average_precision_score(y_oos, prob_oos)
    brier = brier_score_loss(y_oos, prob_oos)
    ci    = bootstrap_metrics(y_oos, prob_oos)

    results.append({
        'Config': name,
        'interactions': params['interactions'],
        'learning_rate': params['learning_rate'],
        'AUROC': round(auroc, 4),
        'AUROC_CI': f'[{ci["auroc_ci"][0]:.3f}, {ci["auroc_ci"][1]:.3f}]',
        'AUPRC': round(auprc, 4),
        'AUPRC_CI': f'[{ci["auprc_ci"][0]:.3f}, {ci["auprc_ci"][1]:.3f}]',
        'Brier': round(brier, 4),
    })
    print(f'  AUROC={auroc:.4f}  AUPRC={auprc:.4f}  Brier={brier:.4f}')
    print()

results_df = pd.DataFrame(results)


## Cell 6 — Summary table and spread diagnostic

In [ ]:
pd.set_option('display.width', 120)
print('=' * 100)
print(' EBM HYPERPARAMETER ROBUSTNESS — STAGE 2 (41-feature M2.5-matched)')
print('=' * 100)
print(results_df.to_string(index=False))
print()

auroc_range = results_df['AUROC'].max() - results_df['AUROC'].min()
auprc_range = results_df['AUPRC'].max() - results_df['AUPRC'].min()

print(f'AUROC range across 5 configs : {auroc_range:.4f}  '
      f'(min={results_df["AUROC"].min():.4f}, max={results_df["AUROC"].max():.4f})')
print(f'AUPRC range across 5 configs : {auprc_range:.4f}  '
      f'(min={results_df["AUPRC"].min():.4f}, max={results_df["AUPRC"].max():.4f})')
print()
print('Interpretation guide: if the range is small relative to the baseline\'s bootstrap CI width,')
print('the baseline result is NOT an artifact of an accidentally well-suited default configuration.')

results_df.to_csv(OUT_DIR / 'nb13_hyperparameter_robustness.csv', index=False)
print(f'\nSaved -> {OUT_DIR / "nb13_hyperparameter_robustness.csv"}')


## Cell 7 — Completion summary

In [ ]:
print('=' * 65)
print(' NB13 — EBM HYPERPARAMETER ROBUSTNESS CHECK COMPLETE')
print('=' * 65)
print()
print('No existing files (NB05A/NB05B/NB12 outputs) were modified.')
print('Next: paste the printed table back to Claude for review before writing into the dissertation.')
